# BRIDGE dataset inspection and cleaning

This notebook loads a fixed BRIDGE snapshot, cleans duplicate and conflicting records, and saves training, validation, and test splits. The preparation code is defined in `scripts/prepare_bridge.py` and uses pandas and NumPy with the local CSV.

The first output shows the dataset revision, row and column counts, column names, and label frequencies. The snapshot contains 19,420 rows and eight columns. Original excerpt text and source metadata are preserved.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from scripts.prepare_bridge import RAW, OUT, REVISION, prepare

print("Pinned revision:", REVISION)
df = pd.read_csv(RAW, dtype=str, keep_default_na=False)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df["bias_label"].value_counts().rename("count").to_frame())


Pinned revision: 645e94fc01424e36b1c7e8f7fe36a9089df96f7f
Raw shape: (19420, 8)
Columns: ['sentence_id', 'text', 'bias_label', 'source', 'register', 'era', 'domain', 'is_augmented']
            count
bias_label       
harmful      7806
harmless     6370
antibias     5244


## Cleaning and split assignment

`prepare()` verifies the raw CSV checksum and normalizes text using Unicode NFKC, case folding, and collapsed whitespace. It removes blank text, excludes all rows in normalized-text groups with conflicting labels, and retains one representative for each remaining normalized text. The cleaning audit records every original row and its disposition.

Punctuation-only variants and excerpts with word-trigram Jaccard similarity of at least 0.90 are grouped together. Whole groups are assigned to approximately 80% training, 10% validation, and 10% test data, balancing class counts with seed 42. This lexical grouping does not identify every paraphrase or shared source document.

The output reports retained and excluded row counts, group sizes, split sizes, and label counts. Cleaning retains 18,560 examples, removes 813 duplicate copies, and excludes 47 rows with conflicting labels.


In [2]:
manifest = prepare()


{
  "rows_raw": 19420,
  "rows_clean": 18560,
  "cleaning_counts": {
    "kept": 18560,
    "normalized_duplicate": 813,
    "label_conflict": 47
  },
  "groups_with_multiple_rows": 299,
  "largest_group": 6,
  "split_rows": {
    "train": 14846,
    "validation": 1857,
    "test": 1857
  },
  "label_counts": {
    "train": {
      "harmful": 6140,
      "harmless": 4676,
      "antibias": 4030
    },
    "validation": {
      "harmful": 768,
      "harmless": 585,
      "antibias": 504
    },
    "test": {
      "harmful": 768,
      "harmless": 585,
      "antibias": 504
    }
  }
}


## Split checks

The code reloads the three saved CSV files and checks that retained IDs and normalized texts are unique, the combined row count matches the cleaning manifest, each group belongs to one split, and every split contains all three labels.

The table shows label counts by split. The splits contain 14,846 training examples, 1,857 validation examples, and 1,857 test examples. The PASS message appears after all assertions succeed.


In [3]:
splits = {name: pd.read_csv(OUT / f"{name}.csv", dtype=str, keep_default_na=False)
          for name in ["train", "validation", "test"]}
combined = pd.concat(splits.values(), ignore_index=True)
assert combined["sentence_id"].is_unique
assert len(combined) == manifest["rows_clean"]
assert combined.groupby("group_id")["split"].nunique().max() == 1
from scripts.prepare_bridge import normalize
assert combined["text"].map(normalize).is_unique
assert all(set(part["bias_label"]) == set(manifest["labels"]) for part in splits.values())
display(pd.crosstab(combined["split"], combined["bias_label"]).reindex(["train", "validation", "test"]))
print("PASS: unique retained IDs/text keys, no group overlap, full coverage, all classes present.")


bias_label  antibias  harmful  harmless
split                                  
train           4030     6140      4676
validation       504      768       585
test             504      768       585
PASS: unique retained IDs/text keys, no group overlap, full coverage, all classes present.


## Split composition and cleaning summary

The tables show example counts by source, register, and augmentation status for each split. Source and label are associated in BRIDGE, and generated examples belong to the antibias class.

The final output summarizes cleaning dispositions, the number of conflicting normalized-text groups, the number of groups with multiple retained rows, the largest group, and the output directory. Full row provenance is stored in `cleaning_audit.csv`; source attribution and license information are recorded in the saved dataset card.


In [4]:
summary = pd.read_csv(OUT / "split_summary.csv")
for field in ["source", "register", "is_augmented"]:
    print(field)
    display(summary.loc[summary["field"].eq(field)].pivot(index="value", columns="split", values="count").fillna(0).astype(int))
print("Cleaning dispositions:", manifest["cleaning_counts"])
print("Conflicting normalized groups:", manifest["conflicting_normalized_groups"])
print("Groups with multiple retained rows:", manifest["groups_with_multiple_rows"])
print("Largest group:", manifest["largest_group"])
print("Output directory:", OUT)


source
split                               test  train  validation
value                                                      
Comparative Agendas Project (CAP)     93    885         127
GovReport Summarization              118    898         118
LSC Eviction Laws Database            24    181          32
Mapping Inequality: Redlining Data   140   1181         168
On the Books: Jim Crow Laws          222   1840         210
Pile of Law                          752   5850         711
Social Bias Frames                   508   4011         491
register
split     test  train  validation
value                            
formal    1349  10835        1366
informal   508   4011         491
is_augmented
split  test  train  validation
value                         
no     1716  13769        1732
yes     141   1077         125
Cleaning dispositions: {'kept': 18560, 'normalized_duplicate': 813, 'label_conflict': 47}
Conflicting normalized groups: 21
Groups with multiple retained rows: 299
Largest